# 06 — Validate the synthetic data

Read-only sanity checks: row counts, uniqueness, foreign keys, value ranges, distributions and the CV PDFs in the Volume.
Each check prints PASS / FAIL; nothing is modified.

In [ ]:
# Environment bootstrap: runs unchanged on Databricks and on a local machine.
# Locally it connects through Databricks Connect (serverless unless DATABRICKS_CLUSTER_ID is set)
# and provides `spark`, `display` and `list_volume_files`. Override the profile with DATABRICKS_CONFIG_PROFILE.
import os
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
LOCAL_SPARK_DEPENDENCIES = []   # pip packages the remote workers need (serverless Databricks Connect only)
if not IS_DATABRICKS:
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "george.sokolovsky2001@gmail.com")
    from databricks.connect import DatabricksSession

    _builder = DatabricksSession.builder
    if os.environ.get("DATABRICKS_CLUSTER_ID"):
        _builder = _builder.clusterId(os.environ["DATABRICKS_CLUSTER_ID"])
    else:
        _builder = _builder.serverless()
        if LOCAL_SPARK_DEPENDENCIES:
            from databricks.connect import DatabricksEnv
            _builder = _builder.withEnvironment(DatabricksEnv().withDependencies(LOCAL_SPARK_DEPENDENCIES))
    spark = _builder.getOrCreate()

    def display(obj, n=1000):
        from IPython.display import display as _display
        _display(obj.limit(n).toPandas() if hasattr(obj, "toPandas") else obj)

def list_volume_files(directory, suffix=""):
    """File names in a Unity Catalog Volume folder (the Volume is only mounted on Databricks, so go through the Files API locally)."""
    if IS_DATABRICKS:
        return [f for f in os.listdir(directory) if f.endswith(suffix)] if os.path.isdir(directory) else []
    from databricks.sdk import WorkspaceClient
    return [e.name for e in WorkspaceClient().files.list_directory_contents(directory) if e.name.endswith(suffix)]


In [0]:
%run ./00_config

In [0]:
# Cell 1 — Hard checks
import os
from pyspark.sql import functions as F

users, details, inc = spark.table(T_USERS), spark.table(T_HANDYMAN_DETAILS), spark.table(T_INCIDENTS)
handy_users = users.where("is_handyman")
results = []
def check(name, ok, detail=""):
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {name}  {detail}")

check("users count (clients + handymen)", users.count() == N_USERS + N_HANDYMEN, f"({users.count():,})")
check("client count",            users.where("NOT is_handyman").count() == N_USERS)
check("handyman users count",    handy_users.count() == N_HANDYMEN)
check("handyman_details count",  details.count() == N_HANDYMEN, f"({details.count():,})")
check("incidents count",         inc.count() >= 1_000_000,       f"({inc.count():,})")
check("user ids unique",         users.select("id").distinct().count() == users.count())
check("required user columns not null", users.where("email IS NULL OR password_hash IS NULL OR first_name IS NULL OR last_name IS NULL OR is_handyman IS NULL OR created_at IS NULL OR is_active IS NULL").count() == 0)

check("handyman_details.user_id unique", details.select("user_id").distinct().count() == details.count())
check("handyman_details FK -> users (is_handyman)", details.join(handy_users.select(F.col("id").alias("u")), details.user_id == F.col("u"), "left_anti").count() == 0)
check("every handyman user has details", handy_users.join(details.select(F.col("user_id").alias("d")), handy_users.id == F.col("d"), "left_anti").count() == 0)
check("handyman rating in 0..5, counts >= 0", details.where("rating_avg < 0 OR rating_avg > 5 OR rating_count < 0 OR completed_cases < 0").count() == 0)
check("handyman rating_count <= completed_cases", details.where("rating_count > completed_cases").count() == 0)

all_emails = users.select("email")
check("emails unique (case-insensitive)", all_emails.select(F.lower("email")).distinct().count() == N_USERS + N_HANDYMEN)
check("emails lowercase and well-formed", all_emails.where(~F.col("email").rlike(r"^[a-z0-9._-]+@[a-z0-9.-]+\.[a-z]{2,}$") | (F.col("email") != F.lower("email"))).count() == 0)
check("emails only on reserved example.* domains", all_emails.where(~F.col("email").rlike(r"@example\.(com|org|net)$")).count() == 0)
check("email length <= 64", all_emails.where(F.length("email") > 64).count() == 0)

addr_cols = ["house", "postal_code", "city"]
check("addresses unique across clients AND handymen", users.select(*addr_cols).distinct().count() == N_USERS + N_HANDYMEN)
check("coordinates inside Germany", users.where(~F.col("latitude").between(47, 55.2) | ~F.col("longitude").between(5.5, 15.5)).count() == 0)

check("incident FK reported_by -> client users", inc.join(users.where("NOT is_handyman").select(F.col("id").alias("u")), inc.reported_by_user_id == F.col("u"), "left_anti").count() == 0)
check("incident FK handyman -> handyman users",  inc.where("handyman_user_id IS NOT NULL").join(handy_users.select(F.col("id").alias("h")), F.col("handyman_user_id") == F.col("h"), "left_anti").count() == 0)
check("incident ids unique",   inc.select("id").distinct().count() == inc.count())
check("status values valid",   inc.where(~F.col("status").isin("open", "recommended", "assigned", "in_progress", "completed", "cancelled")).count() == 0)
check("urgency values valid",  inc.where(~F.col("urgency").isin("low", "medium", "high", "critical")).count() == 0)
check("rating in 1..5",        inc.where("rating IS NOT NULL AND rating NOT BETWEEN 1 AND 5").count() == 0)
check("completed rows have completed_at + handyman", inc.where("status = 'completed' AND (completed_at IS NULL OR handyman_user_id IS NULL)").count() == 0)
check("timestamps ordered",    inc.where("assigned_at < created_at OR completed_at < assigned_at OR updated_at < created_at").count() == 0)
check("incidents created after both accounts exist",
      inc.join(users.select(F.col("id").alias("u"), F.col("created_at").alias("u_created")), inc.reported_by_user_id == F.col("u"))
         .where("created_at < u_created").count() == 0)
check("no timestamp in the future", inc.where("updated_at > current_timestamp()").count() == 0)
check("incident types in taxonomy", inc.where(~F.col("incident_type").isin(*SPECIALISATIONS)).count() == 0)
check("chosen handyman has the specialisation",
      inc.where("handyman_user_id IS NOT NULL").join(details.select(F.col("user_id").alias("h"), "specialisations"), F.col("handyman_user_id") == F.col("h"))
         .where(~F.expr("array_contains(specialisations, incident_type)")).count() == 0)
check("recommended ids are handymen", inc.where("recommended_handyman_ids IS NOT NULL").select(F.explode("recommended_handyman_ids").alias("r")).distinct()
         .join(handy_users.select(F.col("id").alias("h")), F.col("r") == F.col("h"), "left_anti").count() == 0)
check("handyman completed_cases match incidents",
      details.join(inc.where("status = 'completed'").groupBy("handyman_user_id").count(), details.user_id == F.col("handyman_user_id"), "left")
             .where("completed_cases != coalesce(count, 0)").count() == 0)

files = list_volume_files(CV_DIR, ".pdf")
check("one CV PDF per handyman in the Volume", len(files) == N_HANDYMEN, f"({len(files):,} files in {CV_DIR})")
print("\nALL CHECKS PASSED" if all(results) else f"\n{results.count(False)} CHECK(S) FAILED")


In [0]:
# Cell 2 — Distributions (eyeball these)
display(inc.groupBy("status").count().orderBy(F.desc("count")))

In [0]:
display(inc.groupBy("incident_type", "urgency").count().orderBy("incident_type", "urgency"))

In [0]:
display(inc.where("rating IS NOT NULL").groupBy("rating").count().orderBy("rating"))

In [0]:
display(inc.groupBy(F.year("created_at").alias("year")).count().orderBy("year"))

In [0]:
display(inc.where("handyman_user_id IS NOT NULL").agg(F.round(F.avg("distance_km"), 1).alias("avg_km"), F.round(F.avg("travel_time_minutes"), 1).alias("avg_min"),
                                                 F.round(F.max("distance_km"), 1).alias("max_km")))

In [0]:
# Cell 3 — Sample logins (the password is the same for every account)
print(f"Demo password: {DEMO_PASSWORD}")
display(users.where("NOT is_handyman").select("id", "email", "is_handyman").limit(5).unionByName(handy_users.select("id", "email", "is_handyman").limit(5)))
display(inc.where("status = 'completed' AND rating IS NOT NULL").limit(15))

In [0]:
display(details.drop("cv_raw_text").limit(15))